In [2]:
import os
import google.generativeai as genai
from datetime import datetime, timedelta, timezone, UTC
from dotenv import load_dotenv
from pypdf import PdfReader
import json
import ast

/home/robotica/Documentos/IA-minirobots/S7_IA_generativa/Trabajo/IA_minibots_T7_IA_generativa/Gemini/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_28496/3358998624.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
# 1. Definimos las funciones de herramienta (tools) con Type Hints y Docstrings mejorados
def definir_utc(lugar: str) -> str:
    """Obtiene el offset de la zona horaria (UTC) de una ciudad, país o región.
    
    Args: 
        lugar: Nombre del lugar (ej. "Medellín", "Madrid").
    """
    print(f'[Tool] Consultando UTC para: {lugar}')
    # Usar un modelo aislado para la consulta y no interferir con la sesión principal de chat
    temp_model = genai.GenerativeModel('gemini-2.5-flash')
    respuesta = temp_model.generate_content(
        f"Devuelve ÚNICAMENTE el número que representa el offset UTC de la zona horaria actual de {lugar} (en horas). Por ejemplo, si es UTC-5, responde solo '-5'."
    )
    offset = respuesta.text.strip()
    print(f'[Tool] El UTC obtenido para {lugar} es: {offset}')
    return offset

def get_current_time(timezone_offset: float) -> str:
    """Calcula la fecha y hora actual según el offset de una zona horaria.
    Si desconoces el offset para una ciudad, primero llama a la herramienta definir_utc.
    
    Args: 
        timezone_offset: Offset de la zona horaria en horas (ej: -5, 2, 5.5).
    """
    hora_utc = datetime.now(UTC)
    tz_local = timezone(timedelta(hours=timezone_offset))
    hora_local = hora_utc.astimezone(tz_local)
    
    # Calcular horas faltantes para las 6:00 PM (18:00) locales
    hora_6pm = datetime.now(tz_local).replace(hour=18, minute=0, second=0, microsecond=0)
    
    # Si ya pasaron las 6 PM, calculamos para el día siguiente
    if hora_local > hora_6pm:
         hora_6pm += timedelta(days=1)
         
    faltan = (hora_6pm - hora_local).total_seconds() / 3600
    
    res = f"Hora UTC: {hora_utc.strftime('%Y-%m-%d %H:%M:%S')}, Hora local: {hora_local.strftime('%Y-%m-%d %H:%M:%S')}, Faltan: {faltan:.2f} horas para las 6 PM."
    print(f'[Tool] Calculando hora (offset {timezone_offset}): {res}')
    return res

def get_data_dir():
    """Devuelve la ruta correcta de la carpeta de datos."""
    if os.path.exists("data"): return "data"
    if os.path.exists("../data"): return "../data"
    return "data"

def revision_nuevas_fuentes() -> str:
    """Revisa si hay nuevos archivos PDF de fuentes en la carpeta data y actualiza el índice de resúmenes. Ejecuta esto antes de buscar información mediante responder_con_fuente si no estás seguro de conocer todas las fuentes disponibles."""
    data_dir = get_data_dir()
    os.makedirs(data_dir, exist_ok=True)
    indice_path = os.path.join(data_dir, 'indice.json')
    
    if os.path.exists(indice_path):
        try:
            with open(indice_path, 'r') as f:
                contenido = f.read()
                indice = json.loads(contenido) if contenido else {}
        except Exception:
            indice = {}
    else:
        indice = {}
        
    nuevos_archivos = 0
    for i in os.listdir(data_dir):
        if i.endswith(".pdf"):
            if i not in indice:
                print(f"[Tool] Generando e indexando resumen para la nueva fuente: {i}")
                resumen_dict = resumen_fuentes(i)
                if i in resumen_dict:
                    indice.update(resumen_dict)
                    nuevos_archivos += 1

    with open(indice_path, 'w') as f:
        json.dump(indice, f, indent=4)
        
    return f"Se indexaron {nuevos_archivos} nuevas fuentes de información. El índice actual contiene los siguientes archivos: {list(indice.keys())}."

def resumen_fuentes(nombre_archivo: str) -> dict:
    """Genera un resumen interno de un archivo PDF específico para ser referenciado en el índice.
    
    Args: 
        nombre_archivo: Nombre del archivo a resumir, con la extensión .pdf.
    """
    data_dir = get_data_dir()
    ruta = os.path.join(data_dir, nombre_archivo)
    if os.path.exists(ruta):
        try:
            reader = PdfReader(ruta)
            full_text = ""
            # Limitar la extracción de resumen a las primeras 10 páginas para optimizar
            for page in reader.pages[:10]:
                text = page.extract_text()
                if text:
                    full_text += text + "\n"
                    
            temp_model = genai.GenerativeModel('gemini-2.5-flash-lite')
            respuesta = temp_model.generate_content(
                f"Resume el siguiente texto en máximo 100 palabras detallando los conceptos clave para saber de qué trata este documento:\n{full_text}"
            )
            return {nombre_archivo: respuesta.text}
        except Exception as e:
            return {nombre_archivo: f'Error al leer PDF: {e}'}
    else:
        return {nombre_archivo: 'Error: Archivo no encontrado'}

def seleccionar_fuente(pregunta: str) -> str:
    """Selecciona el nombre del archivo PDF en el índice que sea más relevante para responder la pregunta dictada.
    
    Args: 
        pregunta: Pregunta clave original del usuario a resolver.
    """
    data_dir = get_data_dir()
    indice_path = os.path.join(data_dir, 'indice.json')
    
    if os.path.exists(indice_path):
        try:
            with open(indice_path, 'r') as f:
                contenido = f.read()
                indice = json.loads(contenido) if contenido else {}
        except Exception:
            indice = {}
    else:
        return "No se encontró el índice de fuentes. Ejecuta revision_nuevas_fuentes primero."
        
    if indice:
        temp_model = genai.GenerativeModel('gemini-2.5-flash')
        prompt = f"Tenemos los siguientes documentos y sus respectivos resúmenes temáticos:\n{json.dumps(indice, indent=2)}\n\n¿Cuál es el ÚNICO archivo de los enumerados que es el más relevante o habla directamente para dictar una respuesta a la pregunta: '{pregunta}'?\nResponde única y exclusivamente con el nombre exacto del archivo, terminado en .pdf (sin más texto y omitiendo comillas)."
        respuesta = temp_model.generate_content(prompt)
        archivo_seleccionado = respuesta.text.strip()
        print(f"[Tool] Selección de fuente LLM: {archivo_seleccionado}")
        return archivo_seleccionado
    else:
        return "El índice está vacío. Debes ejecutar revision_nuevas_fuentes para escanear archivos PDF."

def responder_con_fuente(pregunta: str) -> str:
    """Responde la pregunta del usuario extrayendo la información directamente iterando sobre el PDF más apropiado en la base de datos indexada.
    
    Args: 
        pregunta: La pregunta completa o detalle de lo que requiere el usuario.
    """
    data_dir = get_data_dir()
    nombre_archivo = seleccionar_fuente(pregunta)
    
    # En caso de que no haya encontrado un archivo válido y el formato falle
    if not nombre_archivo.endswith('.pdf'):
        # Forzar una indexación para asegurarse que no hubo omisiones recientes
        revision_nuevas_fuentes()
        nombre_archivo = seleccionar_fuente(pregunta)
        
    ruta = os.path.join(data_dir, nombre_archivo)
    if os.path.exists(ruta):
        try:
            reader = PdfReader(ruta)
            full_text = ""
            for page in reader.pages:
                 text = page.extract_text()
                 if text:
                     full_text += text + "\n"
            
            # Contexto extendido y exhaustivo para responder en flash model
            temp_model = genai.GenerativeModel('gemini-2.5-flash')
            print(f"[Tool] Estudiando fuente completa para responder usando: {nombre_archivo}")
            respuesta = temp_model.generate_content(
                f"Basándote ÚNICAMENTE en toda la información contextual extraída del documento que cito abajo, responde detallada y extensamente la pregunta del usuario. \nNo incluyas conocimientos externos ajenos al texto.\n\nFuente de Conocimiento ({nombre_archivo}):\n{full_text}\n\nPregunta a resolver: {pregunta}"
            )
            return f"(Respuesta fundamentada en {nombre_archivo}):\n\n{respuesta.text}"
        except Exception as e:
            return f"Hubo un error cargando el PDF '{nombre_archivo}': {e}"
    else:
        return f"Error crítico: No se pudo localizar en disco el archivo '{nombre_archivo}'. Intenta ejecutar revision_nuevas_fuentes y luego responder_con_fuente nuevamente, o reformular la búsqueda."

def listar_fuentes_disponibles() -> str:
    """Lista todos los documentos PDF disponibles actualmente en la base de datos y sus resúmenes. Útil para decirle al usuario qué información exacta hay cargada y disponible para consultar."""
    data_dir = get_data_dir()
    indice_path = os.path.join(data_dir, 'indice.json')
    if os.path.exists(indice_path):
        try:
            with open(indice_path, 'r') as f:
                contenido = f.read()
                indice = json.loads(contenido) if contenido else {}
            if indice:
                return f"Fuentes disponibles en la base de datos:\n{json.dumps(indice, indent=2)}"
        except Exception as e:
            return f"Error al leer el índice: {e}"
    return "No hay fuentes disponibles o el índice no ha sido generado. Deberías sugerir o ejecutar revision_nuevas_fuentes."

def leer_pagina_especifica_pdf(nombre_archivo: str, numero_pagina: int) -> str:
    """Lee y extrae textualmente una página específica de un documento PDF.
    
    Args:
        nombre_archivo: Nombre exacto del archivo PDF (ej: documento.pdf).
        numero_pagina: Número entero de la página a leer (La primera página es 0, la segunda es 1, etc.).
    """
    data_dir = get_data_dir()
    ruta = os.path.join(data_dir, nombre_archivo)
    if os.path.exists(ruta):
        from pypdf import PdfReader
        try:
            reader = PdfReader(ruta)
            if numero_pagina < 0 or numero_pagina >= len(reader.pages):
                return f"Error: La página {numero_pagina} está fuera de rango. El documento tiene un total de {len(reader.pages)} páginas (índices del 0 al {len(reader.pages)-1})."
            text = reader.pages[numero_pagina].extract_text()
            return f"Contenido exacto de la página {numero_pagina} de {nombre_archivo}:\n{text}"
        except Exception as e:
            return f"Error leyendo la página del PDF '{nombre_archivo}': {e}"
    else:
        return f"Error: El archivo '{nombre_archivo}' no existe en la carpeta de datos."



In [4]:
# 2. Clase Agente para encapsular la configuración y ejecución
class AgenteAsistente:
    def __init__(self, model_name: str = 'gemini-2.5-flash'):
        # Cargar variables de entorno (por si hay .env local)
        load_dotenv()
        
        # Validar o asignar la API KEY
        api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("APIKEY")
        if not api_key:
            raise ValueError("No se encontró la API KEY de Gemini. Configura GEMINI_API_KEY o APIKEY en tu entorno o en el archivo .env.")
        
        # Configurar generador de IA
        genai.configure(api_key=api_key)
        
        # Definir la lista de herramientas permitidas para el agente
        self.herramientas = [
            get_current_time, 
            definir_utc,
            revision_nuevas_fuentes,
            resumen_fuentes,
            seleccionar_fuente,
            responder_con_fuente,
            listar_fuentes_disponibles,
            leer_pagina_especifica_pdf
        ]
        
        # Instanciar el modelo con las herramientas habilitadas
        self.model = genai.GenerativeModel(
            model_name=model_name,
            tools=self.herramientas
        )
        
        # Iniciar chat con control automático de herramientas
        self.chat = self.model.start_chat(enable_automatic_function_calling=True)

    def consultar(self, mensaje: str) -> str:
        """Envía un mensaje al agente y obtiene su respuesta usando las herramientas proporcionadas."""
        try:
            print(f"\nUsuario: {mensaje}")
            respuesta = self.chat.send_message(mensaje)
            return respuesta.text
        except Exception as e:
            return f"Hubo un error al comunicarse con el modelo: {e}"

In [ ]:
try:
    # Instanciamos la clase del agente
    agente = AgenteAsistente()
    
    pregunta_2 = "responde con base en las fuentes que es un automata celular"
    resp_2 = agente.consultar(pregunta_2)
    print(f"Agente:\n{resp_2}")
    
except Exception as e:
    print(f"Error de inicialización: {e}")



Usuario: responde con base en las fuentes que es un automata celular
[Tool] Generando e indexando resumen para la nueva fuente: 4. Programacion Genetica 26 1.pdf
[Tool] Generando e indexando resumen para la nueva fuente: 3 Algoritmos Genéticos 26 1.pdf
[Tool] Generando e indexando resumen para la nueva fuente: 6. Machine Learning 26 1.pdf
[Tool] Generando e indexando resumen para la nueva fuente: 5 Redes Neuronales1 2026 1.pdf
[Tool] Generando e indexando resumen para la nueva fuente: 7. Gen Ai 2026 1.pdf
[Tool] Generando e indexando resumen para la nueva fuente: 1. INTRODUCCION IA 26 1.pdf
[Tool] Generando e indexando resumen para la nueva fuente: 2. Autómatas Celulares 26 1.pdf
[Tool] Selección de fuente LLM: 2. Autómatas Celulares 26 1.pdf
[Tool] Estudiando fuente completa para responder usando: 2. Autómatas Celulares 26 1.pdf
Agente:
Un Autómata Celular (AC) es un sistema complejo, discreto y dinámico, concebido originalmente por John von Neumann y Stanisław Ulam en los años 40. L

In [9]:
try:
    # Instanciamos la clase del agente
    agente = AgenteAsistente()
    
    pregunta_2 = "responde con base en las fuentes cuales son los ejercicios propuestos para el documento de automatas celulares"
    resp_2 = agente.consultar(pregunta_2)
    print(f"Agente:\n{resp_2}")
    
except Exception as e:
    print(f"Error de inicialización: {e}")



Usuario: responde con base en las fuentes cuales son los ejercicios propuestos para el documento de automatas celulares
[Tool] Selección de fuente LLM: 2. Autómatas Celulares 26 1.pdf
[Tool] Estudiando fuente completa para responder usando: 2. Autómatas Celulares 26 1.pdf
Agente:
Los ejercicios propuestos en la sección "2.10 Ejercicios y Problemas" del documento "2. Autómatas Celulares 26 1.pdf" son los siguientes:

1.  Observe sus comportamientos en la casa, en la universidad y en el medio de transporte que utiliza. Encuentre, para cada uno de estos escenarios sus reglas básicas.
2.  Suponga una enfermedad, o un incendio forestal, o una moda, desarrolle un modelo de difusión usando Autómatas Celulares probabilísticos. O simule un robot con dos ruedas que evite obstáculos.
3.  Simule el comportamiento de un robot con tres sensores de distancia, que recorre un espacio bidimensional, donde hay 4 objetos distribuidos aleatoriamente, que no se choca con esos objetos.
4.  Tome el plano de 